2025-11-25. Demo of restructured mass spectrum simulation project

This notebook tests my scripts for a small subset of the molecules in my real dataset. 
In the future, we can use this for a workflow demo. 

In [4]:
import os
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Remove duplicates from dataset and write smiles in canonical form. Output: A list of SMILES to use as input for derivitization. 

In [2]:
! . ~/.bashrc
!echo $SCRIPTS_NEIMS/processing

/scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing


In [5]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/remove_duplicate_SMILES_entries.py -i ../data/raw/franklin/goamazon.csv -o ../data/processed/franklin/dataset_unique.csv --index_map_file ../data/processed/franklin/duplicate_mapping.csv  --log_file ../data/processed/franklin/dataset_duplicates.csv


Saved 61 unique molecules to ../data/processed/franklin/dataset_unique.csv
Saved duplicate mapping to ../data/processed/franklin/dataset_duplicates.csv
Saved old->new index mapping to ../data/processed/franklin/duplicate_mapping.csv


## 3. Make molecule folders and run sdf from smiles

In [5]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/make_directories_and_sdfs.py  --input_csv ../data/processed/franklin/dataset_unique.csv --output_root ../data/simulation_results/franklin/QCxMS_10_ps


[WARNING] 'Modified_SMILES' column not found. Using 'SMILES' instead. Derivatized SMILES will not be analyzed.
[NEW] Creating folder: 0000
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0001
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0002
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0003
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0004
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0005
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0006
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0007
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0008
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0009
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0010
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0011
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0012
  → Writi

In [6]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/make_directories_and_sdfs.py  --input_csv ../data/processed/franklin/dataset_unique.csv --output_root ../data/simulation_results/franklin/QCxMS_25_ps


[WARNING] 'Modified_SMILES' column not found. Using 'SMILES' instead. Derivatized SMILES will not be analyzed.
[NEW] Creating folder: 0000
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0001
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0002
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0003
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0004
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0005
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0006
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0007
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0008
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0009
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0010
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0011
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0012
  → Writi

## 4.1 Run the initial QCxMS ground state optimization and MD

In [7]:
# Paths
sim_dir = os.path.abspath(os.path.join(
    os.getcwd(),
    "../data/simulation_results/franklin/QCxMS_25_ps/"
))
script_path = os.path.abspath(os.path.join(
    sim_dir,
    "../../../../src/workflow/submit_qcxms_gs_md_25_ps.sh"  # adjust as needed
))

# Create directory if it doesn't exist
os.makedirs(sim_dir, exist_ok=True)
print(f"Using simulation directory: {sim_dir}")

# Submit SLURM job
result = subprocess.run(
    ["sbatch", "--array=0-60", script_path],
    cwd=sim_dir,  # run from the simulation directory
    capture_output=True,
    text=True
)

# Print SLURM output
print(result.stdout)
print(result.stderr)

Using simulation directory: /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_25_ps
Submitted batch job 31291162




In [8]:
# Paths
sim_dir = os.path.abspath(os.path.join(
    os.getcwd(),
    "../data/simulation_results/franklin/QCxMS_10_ps/"
))
script_path = os.path.abspath(os.path.join(
    sim_dir,
    "../../../../src/workflow/submit_qcxms_gs_md_10_ps.sh"  # adjust as needed
))

# Create directory if it doesn't exist
os.makedirs(sim_dir, exist_ok=True)
print(f"Using simulation directory: {sim_dir}")

# Submit SLURM job
result = subprocess.run(
    ["sbatch", "--array=0-60", script_path],
    cwd=sim_dir,  # run from the simulation directory
    capture_output=True,
    text=True
)

# Print SLURM output
print(result.stdout)
print(result.stderr)

Using simulation directory: /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_10_ps
Submitted batch job 31291230




## 4.2 Run QCxMS fragmentation script for each molecule

In [1]:
import os
import subprocess

# Root simulation directory
sim_dir = "../data/simulation_results/franklin/QCxMS_25_ps"
script_path = os.path.abspath(
    os.path.join(sim_dir, "../../../../src/workflow/submit_qcxms_frag_serial_no_unity.sh")
)

for i in range(0, 61):  # 0000 to 0060 inclusive
    mol = f"{i:04d}"
    mol_path = os.path.join(sim_dir, mol, "GS-opt", "MS-run")

    if not os.path.isdir(mol_path):
        print(f"Skipping {mol}, QCxMS/GS-opt folder not found")
        continue

    result = subprocess.run(
        ["bash", script_path],
        cwd=mol_path,
        capture_output=True,
        text=True
    )

    print(f"Submitted {mol}:")
    print(result.stdout)
    print(result.stderr)


Submitted 0000:
Ground-state run completed successfully: /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_25_ps/0000/GS-opt/MS-run/qcxms.out
Starting parallel QCxMS run on /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_25_ps/0000/GS-opt/MS-run/TMPQCXMS
Detected atom count: 52
Skipping TMP.1 (already finished successfully)
Skipping TMP.10 (already finished successfully)
Skipping TMP.100 (already finished successfully)
Skipping TMP.1000 (already finished successfully)
Skipping TMP.1001 (already finished successfully)
Skipping TMP.1002 (already finished successfully)
Skipping TMP.1003 (already finished successfully)
Skipping TMP.1004 (already finished successfully)
Skipping TMP.1005 (already finished successfully)
Skipping TMP.1006 (already finished successfully)
Skipping TMP.1007 (already finished successfully)
Skipping TMP.1008 (already finished successfully)
Skipping TMP.

In [2]:
import os
import subprocess

# Root simulation directory
sim_dir = "../data/simulation_results/franklin/QCxMS_10_ps"
script_path = os.path.abspath(
    os.path.join(sim_dir, "../../../../src/workflow/submit_qcxms_frag_serial_no_unity.sh")
)

for i in range(0, 61):  # 0000 to 0060 inclusive
    mol = f"{i:04d}"
    mol_path = os.path.join(sim_dir, mol, "GS-opt", "MS-run")

    if not os.path.isdir(mol_path):
        print(f"Skipping {mol}, QCxMS/GS-opt folder not found")
        continue

    result = subprocess.run(
        ["bash", script_path],
        cwd=mol_path,
        capture_output=True,
        text=True
    )

    print(f"Submitted {mol}:")
    print(result.stdout)
    print(result.stderr)


Submitted 0000:
Ground-state run completed successfully: /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_10_ps/0000/GS-opt/MS-run/qcxms.out
Starting parallel QCxMS run on /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/QCxMS_10_ps/0000/GS-opt/MS-run/TMPQCXMS
Detected atom count: 52
Skipping TMP.1 (already finished successfully)
Skipping TMP.10 (already finished successfully)
Skipping TMP.100 (already finished successfully)
Skipping TMP.1000 (already finished successfully)
Skipping TMP.1001 (already finished successfully)
Skipping TMP.1002 (already finished successfully)
Skipping TMP.1003 (already finished successfully)
Skipping TMP.1004 (already finished successfully)
Skipping TMP.1005 (already finished successfully)
Skipping TMP.1006 (already finished successfully)
Skipping TMP.1007 (already finished successfully)
Skipping TMP.1008 (already finished successfully)
Skipping TMP.

## 4.3 Run QCxMS postprocessing: check that number of successful frag. runs was more than 90 %,
## and generate a raw spectra (unfiltered, with floats for m/z numbers)

## 5.1 Run NEIMS for each molecule using the unoptimized SDF (doesnt care of 3D geom)

In [11]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/make_directories_and_sdfs.py  --input_csv ../data/processed/franklin/dataset_unique.csv --output_root ../data/simulation_results/franklin/NEIMS


[WARNING] 'Modified_SMILES' column not found. Using 'SMILES' instead. Derivatized SMILES will not be analyzed.
[NEW] Creating folder: 0000
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0001
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0002
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0003
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0004
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0005
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0006
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0007
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0008
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0009
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0010
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0011
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0012
  → Writi

In [12]:


# -----------------------------
# Configuration
# -----------------------------

sim_dir = os.path.abspath(os.path.join(
    os.getcwd(),
    "../data/simulation_results/franklin/NEIMS/"
))
script_path = os.path.abspath(os.path.join(
    sim_dir,
    "../../../../src/workflow/submit_neims_array.sh"  # adjust as needed
))

# Create directory if it doesn't exist
os.makedirs(sim_dir, exist_ok=True)
print(f"Using simulation directory: {sim_dir}")

# Submit SLURM job
result = subprocess.run(
    ["sbatch", "--array=0-60", script_path],
    cwd=sim_dir,  # run from the simulation directory
    capture_output=True,
    text=True
)



# -----------------------------
# Output
# -----------------------------
if result.returncode != 0:
    print("Submission failed:")
    print(result.stderr)
else:
    print("SLURM submission output:")
    print(result.stdout)


Using simulation directory: /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/data/simulation_results/franklin/NEIMS
SLURM submission output:
Submitted batch job 31295523



## 5.2 Run CFM-ID for compounds it is valid for 

In [6]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/make_directories_and_sdfs.py  --input_csv ../data/processed/franklin/dataset_unique.csv --output_root ../data/simulation_results/franklin/CFMID


[WARNING] 'Modified_SMILES' column not found. Using 'SMILES' instead. Derivatized SMILES will not be analyzed.
[NEW] Creating folder: 0000
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0001
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0002
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0003
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0004
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0005
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0006
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0007
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0008
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0009
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0010
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0011
  → Writing SMILES file
  → Generating SDF
[NEW] Creating folder: 0012
  → Writi

## 6.1 Create a folder with dataset without duplicates and their experimental spectra

In [ ]:
%run /scratch/project_2006752/hsandstr/Project/atmospheric-ms-benchmark/src/processing/copy_unique_folders.py -m ../data/processed/toy_data/duplicate_mapping.csv -s ../data/raw/exp_ms/toy_tms -o ../data/processed/exp/toy_tms_unique


## 6.2 Compare NEIMS and QCxMS spectra to eachother and to the reference spectra (choices, binning all peaks to closest integer, then removing peaks with X % of base peak intensity)

## 6.3 Test only using 20 most strong signals 

## 7. Plot results